# SGA-2025 + ssl-legacysurvey Tutorial

This notebook walks through the end-to-end workflow for running the
[ssl-legacysurvey](https://github.com/georgestein/ssl-legacysurvey) self-supervised
contrastive-learning pipeline on a small subset of SGA-2025 galaxy mosaics.

**Steps covered:**
1. Select N galaxies from the SGA-2025 sample
2. Build the reference catalog (size / isolation cuts)
3. Build HDF5 input files from the on-disk FITS cutouts
4. Run MoCo v2 inference to get embeddings
5. Inspect the results

**Requirements:**
- SGAML Jupyter kernel (see `etc/README.md`)
- Environment variables `SGA_DIR` and `SGA_DATA_DIR` set (e.g., via `source bin/SGA2025/SGA2025-env`)
- Pre-trained checkpoint `resnet50.ckpt` (download from the ssl-legacysurvey repo)

## 1. Setup

In [ ]:
import os
import numpy as np
from astropy.table import vstack

from SGA.SGA import read_sga_sample
from SGA.ssl import build_ssl_legacysurvey_refcat, build_ssl_legacysurvey, ssl_match
from SGA.logger import log

# Verify required environment variables are set
for var in ('SGA_DIR', 'SGA_DATA_DIR'):
    val = os.getenv(var)
    if val is None:
        raise EnvironmentError(f'{var} is not set. Run: source bin/SGA2025/SGA2025-env')
    print(f'{var} = {val}')

In [3]:
# --- User-settable parameters ---
N_GALAXIES   = 100          # number of galaxies to select
SEED         = 42           # random seed for reproducibility
SSL_VERSION  = 'v3'         # 'v1'/'v2' = grz only; 'v3'/'v4' = grz + griz
CHECKPOINT   = 'resnet50.ckpt'   # path to pre-trained MoCo v2 checkpoint
OUTDIR       = os.path.join(os.environ['PSCRATCH'], 'sga-ssl-tutorial')  # output directory for HDF5 files and results
CUTOUTDIR    = os.environ['SGA_DATA_DIR']  # root of on-disk FITS cutouts

os.makedirs(OUTDIR, exist_ok=True)
print(f'Output directory: {OUTDIR}')

Output directory: /pscratch/sd/i/ioannis/sga-ssl-tutorial


## 2. Load the SGA-2025 sample

`read_sga_sample` returns two tables: `sample` (one row per galaxy group, GROUP_PRIMARY only)
and `fullsample` (all group members including satellites).

In [ ]:
sample_n, fullsample_n = read_sga_sample(region='dr11-north')
sample_s, fullsample_s = read_sga_sample(region='dr11-south')

sample     = vstack([sample_n,     sample_s])
fullsample = vstack([fullsample_n, fullsample_s])
del sample_n, sample_s, fullsample_n, fullsample_s

print(f'Primary sample:  {len(sample):,} groups')
print(f'Full sample:     {len(fullsample):,} objects (including satellites)')

In [11]:
sample

SGAID,SGAGROUP,REGION,OBJNAME,PGC,SAMPLE,ELLIPSEMODE,FITMODE,BX_INIT,BY_INIT,RA_INIT,DEC_INIT,SMA_INIT,DIAM_INIT,BA_INIT,PA_INIT,MAG_INIT,DIAM_INIT_REF,EBV,GROUP_NAME,GROUP_MULT,GROUP_PRIMARY,GROUP_RA,GROUP_DEC,GROUP_DIAMETER,PSFSIZE_G,PSFSIZE_R,PSFSIZE_I,PSFSIZE_Z,PSFDEPTH_G,PSFDEPTH_R,PSFDEPTH_I,PSFDEPTH_Z,BANDS,OPTFLUX,SGANAME,RA,DEC,BX,BY,SMA_MASK,SMA_MOMENT,BA_MOMENT,PA_MOMENT,RA_TRACTOR,DEC_TRACTOR,ELLIPSEBIT,MW_TRANSMISSION_G,MW_TRANSMISSION_R,MW_TRANSMISSION_I,MW_TRANSMISSION_Z,MW_TRANSMISSION_W1,MW_TRANSMISSION_W2,MW_TRANSMISSION_W3,MW_TRANSMISSION_W4,MW_TRANSMISSION_FUV,MW_TRANSMISSION_NUV,GINI_G,GINI_R,GINI_I,GINI_Z,GINI_W1,GINI_W2,GINI_W3,GINI_W4,GINI_FUV,GINI_NUV,COG_MTOT_G,COG_MTOT_R,COG_MTOT_I,COG_MTOT_Z,COG_MTOT_W1,COG_MTOT_W2,COG_MTOT_W3,COG_MTOT_W4,COG_MTOT_FUV,COG_MTOT_NUV,COG_MTOT_ERR_G,COG_MTOT_ERR_R,COG_MTOT_ERR_I,COG_MTOT_ERR_Z,COG_MTOT_ERR_W1,COG_MTOT_ERR_W2,COG_MTOT_ERR_W3,COG_MTOT_ERR_W4,COG_MTOT_ERR_FUV,COG_MTOT_ERR_NUV,COG_DMAG_G,COG_DMAG_R,COG_DMAG_I,COG_DMAG_Z,COG_DMAG_W1,COG_DMAG_W2,COG_DMAG_W3,COG_DMAG_W4,COG_DMAG_FUV,COG_DMAG_NUV,COG_DMAG_ERR_G,COG_DMAG_ERR_R,COG_DMAG_ERR_I,COG_DMAG_ERR_Z,COG_DMAG_ERR_W1,COG_DMAG_ERR_W2,COG_DMAG_ERR_W3,COG_DMAG_ERR_W4,COG_DMAG_ERR_FUV,COG_DMAG_ERR_NUV,COG_LNALPHA1_G,COG_LNALPHA1_R,COG_LNALPHA1_I,COG_LNALPHA1_Z,COG_LNALPHA1_W1,COG_LNALPHA1_W2,COG_LNALPHA1_W3,COG_LNALPHA1_W4,COG_LNALPHA1_FUV,COG_LNALPHA1_NUV,COG_LNALPHA1_ERR_G,COG_LNALPHA1_ERR_R,COG_LNALPHA1_ERR_I,COG_LNALPHA1_ERR_Z,COG_LNALPHA1_ERR_W1,COG_LNALPHA1_ERR_W2,COG_LNALPHA1_ERR_W3,COG_LNALPHA1_ERR_W4,COG_LNALPHA1_ERR_FUV,COG_LNALPHA1_ERR_NUV,COG_LNALPHA2_G,COG_LNALPHA2_R,COG_LNALPHA2_I,COG_LNALPHA2_Z,COG_LNALPHA2_W1,COG_LNALPHA2_W2,COG_LNALPHA2_W3,COG_LNALPHA2_W4,COG_LNALPHA2_FUV,COG_LNALPHA2_NUV,COG_LNALPHA2_ERR_G,COG_LNALPHA2_ERR_R,COG_LNALPHA2_ERR_I,COG_LNALPHA2_ERR_Z,COG_LNALPHA2_ERR_W1,COG_LNALPHA2_ERR_W2,COG_LNALPHA2_ERR_W3,COG_LNALPHA2_ERR_W4,COG_LNALPHA2_ERR_FUV,COG_LNALPHA2_ERR_NUV,COG_CHI2_G,COG_CHI2_R,COG_CHI2_I,COG_CHI2_Z,COG_CHI2_W1,COG_CHI2_W2,COG_CHI2_W3,COG_CHI2_W4,COG_CHI2_FUV,COG_CHI2_NUV,COG_NDOF_G,COG_NDOF_R,COG_NDOF_I,COG_NDOF_Z,COG_NDOF_W1,COG_NDOF_W2,COG_NDOF_W3,COG_NDOF_W4,COG_NDOF_FUV,COG_NDOF_NUV,SMA50_G,SMA50_R,SMA50_I,SMA50_Z,SMA50_W1,SMA50_W2,SMA50_W3,SMA50_W4,SMA50_FUV,SMA50_NUV,SMA50_ERR_G,SMA50_ERR_R,SMA50_ERR_I,SMA50_ERR_Z,SMA50_ERR_W1,SMA50_ERR_W2,SMA50_ERR_W3,SMA50_ERR_W4,SMA50_ERR_FUV,SMA50_ERR_NUV,SMA_AP00,SMA_AP01,SMA_AP02,SMA_AP03,SMA_AP04,FLUX_AP00_G,FLUX_AP00_R,FLUX_AP00_I,FLUX_AP00_Z,FLUX_AP00_W1,FLUX_AP00_W2,FLUX_AP00_W3,FLUX_AP00_W4,FLUX_AP00_FUV,FLUX_AP00_NUV,FLUX_ERR_AP00_G,FLUX_ERR_AP00_R,FLUX_ERR_AP00_I,FLUX_ERR_AP00_Z,FLUX_ERR_AP00_W1,FLUX_ERR_AP00_W2,FLUX_ERR_AP00_W3,FLUX_ERR_AP00_W4,FLUX_ERR_AP00_FUV,FLUX_ERR_AP00_NUV,FMASKED_AP00_G,FMASKED_AP00_R,FMASKED_AP00_I,FMASKED_AP00_Z,FMASKED_AP00_W1,FMASKED_AP00_W2,FMASKED_AP00_W3,FMASKED_AP00_W4,FMASKED_AP00_FUV,FMASKED_AP00_NUV,FLUX_AP01_G,FLUX_AP01_R,FLUX_AP01_I,FLUX_AP01_Z,FLUX_AP01_W1,FLUX_AP01_W2,FLUX_AP01_W3,FLUX_AP01_W4,FLUX_AP01_FUV,FLUX_AP01_NUV,FLUX_ERR_AP01_G,FLUX_ERR_AP01_R,FLUX_ERR_AP01_I,FLUX_ERR_AP01_Z,FLUX_ERR_AP01_W1,FLUX_ERR_AP01_W2,FLUX_ERR_AP01_W3,FLUX_ERR_AP01_W4,FLUX_ERR_AP01_FUV,FLUX_ERR_AP01_NUV,FMASKED_AP01_G,FMASKED_AP01_R,FMASKED_AP01_I,FMASKED_AP01_Z,FMASKED_AP01_W1,FMASKED_AP01_W2,FMASKED_AP01_W3,FMASKED_AP01_W4,FMASKED_AP01_FUV,FMASKED_AP01_NUV,FLUX_AP02_G,FLUX_AP02_R,FLUX_AP02_I,FLUX_AP02_Z,FLUX_AP02_W1,FLUX_AP02_W2,FLUX_AP02_W3,FLUX_AP02_W4,FLUX_AP02_FUV,FLUX_AP02_NUV,FLUX_ERR_AP02_G,FLUX_ERR_AP02_R,FLUX_ERR_AP02_I,FLUX_ERR_AP02_Z,FLUX_ERR_AP02_W1,FLUX_ERR_AP02_W2,FLUX_ERR_AP02_W3,FLUX_ERR_AP02_W4,FLUX_ERR_AP02_FUV,FLUX_ERR_AP02_NUV,FMASKED_AP02_G,FMASKED_AP02_R,FMASKED_AP02_I,FMASKED_AP02_Z,FMASKED_AP02_W1,FMASKED_AP02_W2,FMASKED_AP02_W3,FMASKED_AP02_W4,FMASKED_AP02_FUV,FMASKED_AP02_NUV,FLUX_AP03_G,FLUX_AP03_R,FLUX_AP03_I,FLUX_AP03_Z,FLUX_AP03_W1,FLUX_AP03_W2,FLUX_AP03_W3,FLUX_AP03_W4,FLUX_AP03_FUV,FLUX_AP03_NUV,FLUX_ERR_AP03_G,FLUX_ERR_AP03_R,FLUX_ERR_AP03_I,FLUX_ERR_AP03_Z,

## 3. Build the ssl reference catalog

Apply size, isolation, and imaging-quality cuts to define the pool of
objects that can serve as *reference* galaxies for the ssl pipeline.
`build_ssl_legacysurvey_refcat` returns:
- `refcat` — objects that pass all reference-galaxy cuts
- `cat`    — objects that pass the looser candidate cuts

In [12]:
refcat, cat = build_ssl_legacysurvey_refcat(sample, fullsample, ssl_version=SSL_VERSION)
print(f'Reference catalog: {len(refcat):,} objects')
print(f'Candidate catalog: {len(cat):,} objects')

INFO:ssl.py:145:build_ssl_legacysurvey_refcat: ssl_version=v3: 13,289 reference objects, 14 candidates to classify.
Reference catalog: 13,289 objects
Candidate catalog: 14 objects


## 4. Select N galaxies for the tutorial

For this tutorial we draw a small random subset so the HDF5 build and
inference steps run quickly.  Modify `select_sample` to apply whatever
science-driven selection you need (e.g., magnitude, colour, environment).

In [14]:
def select_sample(cat, n=100, seed=42):
    """Return a random draw of n objects from cat.

    Parameters
    ----------
    cat : astropy.table.Table
        Input catalog (e.g. the refcat or candidate cat from build_ssl_legacysurvey_refcat).
    n : int
        Number of objects to draw.  If n >= len(cat), returns the full catalog.
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    astropy.table.Table
        Sub-table of n randomly selected rows.
    """
    rng = np.random.default_rng(seed)
    n   = min(n, len(cat))
    idx = rng.choice(len(cat), size=n, replace=False)
    idx.sort()  # preserve spatial ordering
    return cat[idx]


sub = select_sample(cat, n=N_GALAXIES, seed=SEED)
print(f'Selected {len(sub)} galaxies for the tutorial')
sub[['SGAID', 'GROUP_NAME', 'GROUP_RA', 'GROUP_DEC', 'D26', 'BANDS']][:5]

Selected 14 galaxies for the tutorial


SGAID,GROUP_NAME,GROUP_RA,GROUP_DEC,D26,BANDS
int64,str10,float64,float64,float64,str4
892336,20056p6502,200.56548782934075,65.02441410914057,0.1775445193052292,grz
5053877,24344p5436,243.44775420694592,54.36986876331912,0.06323281675577164,grz
5053913,14845p6897,148.45490134288872,68.97051525948696,0.3284244239330292,grz
5053949,15661p6765,156.61884946867596,67.65480534457802,0.05424872785806656,grz
5054095,17947p5661,179.47086337508873,56.61360918438806,0.15327098965644836,grz


## 5. Build HDF5 input files

`build_ssl_legacysurvey` locates the rescaled FITS cutout for each object,
selects the requested band planes, and packs everything into chunked HDF5
files ready for inference.

In [15]:
build_ssl_legacysurvey(
    sub,
    refcat,
    ssl_version=SSL_VERSION,
    cutoutdir=CUTOUTDIR,
    outdir=OUTDIR,
    cutout_regions=['dr11-north', 'dr11-south'],
    verbose=True,
    overwrite=True,
)

DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=1670
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=3139167
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=3140545
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=3142586
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=3143301
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=3144286
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=3144499
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=3144581
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=3145578
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=3146734
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=3146850
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=3147355
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=3147365
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=3151338
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=3151797
DEBUG:ssl.py:202:collect_fil

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4198802
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4199288
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4199353
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4200199
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4201294
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4205386
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4215570
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4221263
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4235164
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4235208
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4235213
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4235221
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4235222
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4235269
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4238994
DEBUG:ssl.py:202:collect_

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4833903
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4833990
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4834022
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4834047
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4834209
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4834254
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4834388
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4834523
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4834569
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4834580
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4834607
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4834650
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4834692
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4834744
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4834772
DEBUG:ssl.py:202:collect_

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4995592
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4995598
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4995619
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4995742
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4995781
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4995895
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4995950
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4995980
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4996057
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4996126
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4996141
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4996148
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4996175
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4996179
DEBUG:ssl.py:202:collect_files: Missing cutout for SGAID=4996191
DEBUG:ssl.py:202:collect_

OSError: [Errno 30] Read-only file system: '/dvs_ro/cfs/cdirs/cosmo/work/legacysurvey/sga/2025/ssl'

In [ ]:
import glob
hdf5_files = sorted(glob.glob(os.path.join(OUTDIR, '*.hdf5')))
print(f'HDF5 files written: {len(hdf5_files)}')
for f in hdf5_files:
    print(' ', f)

## 6. Inspect HDF5 contents

In [ ]:
import h5py
import matplotlib.pyplot as plt

if hdf5_files:
    with h5py.File(hdf5_files[0], 'r') as f:
        print('Keys:', list(f.keys()))
        images = f['images'][:]   # (N, nband, H, W)
        ras    = f['ra'][:]
        decs   = f['dec'][:]
        rows   = f['row'][:]      # SGAID
        print(f'Image array shape: {images.shape}')
        print(f'RA range: {ras.min():.3f} – {ras.max():.3f}')

    # Show the first 9 galaxies (median-scale each for display)
    fig, axes = plt.subplots(3, 3, figsize=(9, 9))
    for ax, img, ra, dec in zip(axes.flat, images[:9], ras[:9], decs[:9]):
        rgb = np.dstack([img[2], img[1], img[0]])  # z, r, g → RGB
        lo, hi = np.percentile(rgb, [1, 99])
        rgb = np.clip((rgb - lo) / (hi - lo + 1e-6), 0, 1)
        ax.imshow(rgb, origin='lower')
        ax.set_title(f'({ra:.3f}, {dec:.3f})', fontsize=7)
        ax.axis('off')
    plt.suptitle('First 9 tutorial galaxies (grz)', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print('No HDF5 files found — check CUTOUTDIR and that cutouts exist on disk.')

## 7. Run ssl_match inference

`ssl_match` loads the pre-trained MoCo v2 backbone, encodes every galaxy
in the HDF5 file, and writes two outputs to `output_dir`:
- `*_output.txt` — per-galaxy embedding vectors
- `*_umap.npy`   — 2-d UMAP projection for visualisation

In [ ]:
results_dir = os.path.join(OUTDIR, 'results')
os.makedirs(results_dir, exist_ok=True)

for path in hdf5_files:
    print(f'Running ssl_match on {os.path.basename(path)} ...')
    ssl_match(
        path,
        checkpoint_path=CHECKPOINT,
        output_dir=results_dir,
        similarity=True,
        threshold=0.5,
    )

## 8. Inspect results

The UMAP projection gives a 2-d view of the embedding space where
morphologically similar galaxies cluster together.

In [ ]:
umap_files = sorted(glob.glob(os.path.join(results_dir, '*_umap.npy')))
print(f'UMAP files: {umap_files}')

if umap_files:
    umap = np.load(umap_files[0])
    print(f'UMAP shape: {umap.shape}')   # (N, 2)

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(umap[:, 0], umap[:, 1], s=8, alpha=0.7, linewidths=0)
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    ax.set_title(f'ssl-legacysurvey embeddings — {len(umap)} galaxies')
    plt.tight_layout()
    plt.show()
else:
    print('No UMAP output found — did ssl_match run successfully?')